<a href="https://colab.research.google.com/github/yakdudi12/IID/blob/main/notebooks/TrabajoFinal_IID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Declaración de uso de IA:
  Se uso claude Opus 5.5 para ver si la base de datos propuesta cumple con los requisitos para poder realizar el Trabajo Práctico.
  Answer: "Si [...]"

Segundo Prompt: "ES muy pesada la base de datos de clinvar, necesito hacer algo on-demand u online, porque se cierra la sesión de colab".
EL reto de las consultas fueron por el problema que es muy pesada la base de datos y se busca resolver ese problema

Tercer Prompt: SE le pidio a CLaude Opus 5.5 que me liste los requisitos que debo cumplir para el Trabajo final quede acorde. Leyo mi carpeta de la materia completa y todos los trabajos realizados en ella. Y el enunciado del Trabajo. Respuesta: Me dio el checklist de cosas que deberia hacer para llegar hasta 2NF, y algunas recomendaciones.

4rto Prompt: Se le pidio a Claude Opus 5.5 que evalue como se distribuyen los nulls dentro del conjunto de datos, y que devuelva esa información junto a una tabla para tomar decisiones de como resolverlos para cada caso especifico.
  4.2: "❯ Volveme a hacer esta tabla con todas las columnas que poseen nulls explicito, implicitos, estructurales y numéricos, con una breve descripcion del por que consideras
  que son nulls y tu propuesta. Asi tomamos decisiones en conjunto sobre como tratar cada uno de estos casos"

5to prompt: "❯ Ahora tendremos que abordar como elegir la mejor candidata a Primary Key (PK), yo ya previamente habia elegido una en otra iteracion, pero tenemos que
  hacerlo de manera formal ahora (En esa ocasion habia elegido una Composite Key que combinaba el UniProtID o en este caso el "NM" junto con otras
  columnas). Primero Barajemos las SUper Key y determinemos las Candidate Keys a usar, y de ahi elegimos una Primary Key formalmente."

6to prompt: A OPus 5.5 le pase mis requisitos para sea 1NF, y le pregunte si los cumpliamos y que nos faltaba solucionar. Se solucionaron esos hechos, y debido a que nos quedamos con 1 sola PK, 2NF tambien quedo asegurado.

In [12]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [13]:
import polars as pl

url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/archive/variant_summary_2026-10.txt.gz"

# Columnas a conservar y su tipo (downcasting: Int32/Int16/Int8 donde el rango alcanza,
# Categorical para texto de baja cardinalidad). El resto de las 43 columnas no se lee.
COLS = {
    "#AlleleID": pl.Int32, "VariationID": pl.Int32, "Type": pl.Categorical, "Name": pl.String,
    "GeneID": pl.Int32, "GeneSymbol": pl.String, "HGNC_ID": pl.String,
    "ClinicalSignificance": pl.Categorical, "ClinSigSimple": pl.Int8, "LastEvaluated": pl.String,
    "PhenotypeIDS": pl.String, "PhenotypeList": pl.String,
    "Origin": pl.Categorical, "OriginSimple": pl.Categorical,
    "Chromosome": pl.Categorical, "Cytogenetic": pl.Categorical, "Start": pl.Int32, "Stop": pl.Int32,
    "ReviewStatus": pl.Categorical, "NumberSubmitters": pl.Int16,
}

df = (pl.scan_csv(url, separator="\t", null_values=["na", "-"], infer_schema=False,
                  schema_overrides={**COLS, "Assembly": pl.Categorical})
        .filter(pl.col("Assembly") == "GRCh38")   # cada variante aparece una vez por ensamblado
        .select(list(COLS))                        # Assembly se usa solo para filtrar
        .with_columns(pl.col("LastEvaluated").str.strptime(pl.Date, "%b %d, %Y"))
        .collect(engine="streaming"))

df.glimpse()

Rows: 4578190
Columns: 20
$ #AlleleID             <i32> 15041, 15042, 15043, 15044, 15045, 15046, 15048, 15049, 15050, 15051
$ VariationID           <i32> 2, 3, 4, 5, 6, 214885, 9, 10, 11, 12
$ Type                  <cat> Indel, Deletion, single nucleotide variant, single nucleotide variant, single nucleotide variant, single nucleotide variant, single nucleotide variant, single nucleotide variant, single nucleotide variant, single nucleotide variant
$ Name                  <str> 'NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer)', 'NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs)', 'NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg)', 'NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter)', 'NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser)', 'NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg)', 'NM_000410.4(HFE):c.845G>A (p.Cys282Tyr)', 'NM_000410.4(HFE):c.187C>G (p.His63Asp)', 'NM_000410.4(HFE):c.193A>T (p.Ser65Cys)', 'NM_000410.4(HFE):c.314T>C (p.Ile105Thr)'
$ GeneID             

# Selección de columnas y tipos

In [14]:
print(f"Memoria del DataFrame: {df.estimated_size('mb'):,.0f} MB")
df["Type"].value_counts(sort=True)

Memoria del DataFrame: 870 MB


Type,count
cat,u32
"""single nucleotide variant""",4238174
"""Deletion""",173701
"""Duplication""",73911
"""Microsatellite""",40521
"""Indel""",19927
"""Insertion""",14969
"""copy number gain""",7450
"""copy number loss""",7234
"""Inversion""",1557


## Motivos: columnas descartadas

| Columna | Justificación |
|---|---|
| `Assembly` | Debido a que solo se uso para filtrar GRCh38 |
| `nsv/esv (dbVar)` | Es un ID externo, con exceso de nulls |
| `RCVaccession` | ID multivalor que posee informacion redudante a otras columnas (PhenotypeList) |
| `ChromosomeAccession` | Redudante si ya fijamos GRCh38 |
| `ReferenceAllele` | No tiene datos |
| `AlternateAllele` | No tiene datos |
| `PositionVCF` | VCF es otro tipo de anotacion de variantes la cual no usamos en este analisis |
| `ReferenceAlleleVCF` | Relacionado con PositionVCF |
| `AlternateAlleleVCF` | Relacionado con PositionVCF |
| `Guidelines` | No se usa |
| `TestedInGTR` | No se usa |
| `OtherIDs` | IDs en otras bases de datos, es multivalor. Pero no lo necesitamos |
| `SubmitterCategories` | NO se usa |
| `SomaticClinicalImpact` | Sin datos |
| `SomaticClinicalImpactLastEvaluated` | NO se usa |
| `ReviewStatusClinicalImpact` | NO se usa |
| `Oncogenicity` | Aplica a solo 1 tipo de variantes |
| `OncogenicityLastEvaluated` | Lo mismo de arriba |
| `ReviewStatusOncogenicity` | Lo mismo de arriba |
| `SCVsForAggregateGermlineClassification` | Redundante (CLinicalSIgnificance y NUmberSUbmitters)|
| `SCVsForAggregateSomaticClinicalImpact` | LO mismo |
| `SCVsForAggregateOncogenicityClassification` | Lo mismo |
| `RS# (dbSNP)` | |


# Valores faltantes

## 1. Nulls explícitos

In [15]:
pl.Config.set_tbl_rows(25)
n = df.height
(df.null_count()
   .transpose(include_header=True, header_name="columna", column_names=["nulos"])
   .with_columns((pl.col("nulos") / n * 100).round(2).alias("pct"))
   .sort("nulos", descending=True))

columna,nulos,pct
str,u32,f64
"""LastEvaluated""",276994,6.05
"""ClinicalSignificance""",244562,5.34
"""ReviewStatus""",244562,5.34
"""PhenotypeIDS""",73955,1.62
"""HGNC_ID""",28729,0.63
"""Cytogenetic""",3140,0.07
"""GeneSymbol""",1661,0.04
"""PhenotypeList""",767,0.02
"""#AlleleID""",0,0.0


## 2. Nulls ocultos (valores centinela)

ClinVar codifica la ausencia de dato con valores que no son null: `-1`, `0` o textos como *not provided*.

In [16]:
SIN_CLASIF = ["not provided", "no classification for the single variant",
              "no classifications from unflagged records"]
SIN_FENO = ["not provided", "not specified"]

(df.select(
    (pl.col("GeneID") == -1).sum().alias("GeneID = -1"),
    (pl.col("ClinSigSimple") == -1).sum().alias("ClinSigSimple = -1"),
    (pl.col("NumberSubmitters") == 0).sum().alias("NumberSubmitters = 0"),
    pl.col("ClinicalSignificance").cast(pl.String).is_in(SIN_CLASIF).sum().alias("ClinicalSignificance sin clasificación"),
    pl.col("PhenotypeList").str.split("|").list.eval(pl.element().is_in(SIN_FENO)).list.all().sum()
      .alias("PhenotypeList solo not provided/specified"),
    pl.col("OriginSimple").cast(pl.String).is_in(["not provided", "unknown", "not applicable"]).sum()
      .alias("OriginSimple sin dato"),
  )
  .transpose(include_header=True, header_name="centinela", column_names=["filas"])
  .with_columns((pl.col("filas") / n * 100).round(2).alias("pct")))

centinela,filas,pct
str,u32,f64
"""GeneID = -1""",28174,0.62
"""ClinSigSimple = -1""",245244,5.36
"""NumberSubmitters = 0""",682,0.01
"""ClinicalSignificance sin clasi…",7952,0.17
"""PhenotypeList solo not provide…",2156750,47.11
"""OriginSimple sin dato""",305239,6.67


## 3. ¿Los nulls de clasificación aparecen juntos?

Si varias columnas son nulas siempre en las mismas filas, el faltante es estructural (el registro no tiene esa información por diseño) y no un error de carga.

In [17]:
(df.group_by(
    pl.col("ClinicalSignificance").is_null().alias("ClinSig_null"),
    pl.col("ReviewStatus").is_null().alias("Review_null"),
    (pl.col("ClinSigSimple") == -1).alias("ClinSigSimple_-1"),
    (pl.col("NumberSubmitters") == 0).alias("Submitters_0"),
    pl.col("LastEvaluated").is_null().alias("Fecha_null"),
  ).len().sort("len", descending=True))

ClinSig_null,Review_null,ClinSigSimple_-1,Submitters_0,Fecha_null,len
bool,bool,bool,bool,bool,u32
false,false,false,false,false,4301196
true,true,true,false,true,244562
false,false,false,false,true,31750
false,false,true,true,true,682


In [18]:
# Origen de las variantes sin clasificación germinal
df.filter(pl.col("ClinicalSignificance").is_null())["OriginSimple"].value_counts(sort=True)

OriginSimple,count
cat,u32
"""not applicable""",242968
"""somatic""",1567
"""germline/somatic""",25
"""unknown""",2


## 4. Faltantes por tipo de variante

In [19]:
(df.group_by("Type").agg(
    pl.len().alias("n"),
    (pl.col("GeneID") == -1).mean().mul(100).round(1).alias("% GeneID=-1"),
    pl.col("Cytogenetic").is_null().mean().mul(100).round(1).alias("% Cytogenetic null"),
    pl.col("LastEvaluated").is_null().mean().mul(100).round(1).alias("% LastEvaluated null"),
    pl.col("ClinicalSignificance").is_null().mean().mul(100).round(1).alias("% ClinSig null"),
  ).sort("n", descending=True))

Type,n,% GeneID=-1,% Cytogenetic null,% LastEvaluated null,% ClinSig null
cat,u32,f64,f64,f64,f64
"""single nucleotide variant""",4238174,0.3,0.1,6.1,5.6
"""Deletion""",173701,1.0,0.0,7.7,4.6
"""Duplication""",73911,1.1,0.0,3.2,0.3
"""Microsatellite""",40521,0.8,0.0,3.6,1.4
"""Indel""",19927,0.4,0.0,2.8,0.5
"""Insertion""",14969,0.9,0.2,5.0,1.2
"""copy number gain""",7450,92.3,0.0,0.8,0.0
"""copy number loss""",7234,87.6,0.0,1.9,0.0
"""Inversion""",1557,0.6,0.3,1.3,0.3


## 5. ¿Se pueden imputar?

In [20]:
# GeneSymbol / HGNC_ID: ¿el GeneID de esas filas tiene símbolo/HGNC en otra fila?
print("GeneSymbol null con GeneID != -1:",
      df.filter(pl.col("GeneSymbol").is_null() & (pl.col("GeneID") != -1)).height)
hgnc_null = df.filter(pl.col("HGNC_ID").is_null() & (pl.col("GeneID") != -1))
con_hgnc = df.filter(pl.col("HGNC_ID").is_not_null()).select("GeneID").unique()
print("HGNC_ID null con GeneID != -1:", hgnc_null.height,
      "| imputables desde otra fila:", hgnc_null.join(con_hgnc, on="GeneID").height)

# Cytogenetic: ¿en qué cromosomas falta?
print(df.filter(pl.col("Cytogenetic").is_null())["Chromosome"].value_counts(sort=True))

# PhenotypeIDS null con PhenotypeList presente
print(df.filter(pl.col("PhenotypeIDS").is_null() & pl.col("PhenotypeList").is_not_null())
        ["PhenotypeList"].value_counts(sort=True).head(5))

GeneSymbol null con GeneID != -1: 0
HGNC_ID null con GeneID != -1: 555 | imputables desde otra fila: 9
shape: (6, 2)
┌────────────┬───────┐
│ Chromosome ┆ count │
│ ---        ┆ ---   │
│ cat        ┆ u32   │
╞════════════╪═══════╡
│ MT         ┆ 3133  │
│ 9          ┆ 2     │
│ 6          ┆ 2     │
│ 16         ┆ 1     │
│ 22         ┆ 1     │
│ Un         ┆ 1     │
└────────────┴───────┘
shape: (5, 2)
┌─────────────────────────────────┬───────┐
│ PhenotypeList                   ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ See cases                       ┆ 14693 │
│ Becker muscular dystrophy, Car… ┆ 581   │
│ PLXNA4-related disorder         ┆ 525   │
│ EBV-positive nodal T- and NK-c… ┆ 493   │
│ PKD1-related disorder           ┆ 488   │
└─────────────────────────────────┴───────┘


## Decisiones sobre valores faltantes

Los conteos de filas son los que afecta cada decisión, aplicadas en este orden.

| # | Columna | Caso | Filas | Decisión |
|---|---|---|---|---|
| 1 | `ClinSigSimple`, `ClinicalSignificance`, `ReviewStatus` | Sin clasificación germinal: `ClinSigSimple = -1` y las otras dos en null | 245.244 | **Filas descartadas.** Los tres faltantes aparecen siempre juntos y el 99 % son variantes de origen *not applicable*: solo tienen clasificación somática u oncogénica, que quedó fuera del análisis. Sin clasificación germinal no aportan al objetivo del trabajo. |
| 2 | `ClinicalSignificance`, `ReviewStatus` | Textos que significan "sin dato": *not provided*, *no classification provided*, *no classifications from unflagged records* | 6.746 | **Unificados en "Sin clasificación".** Son nulls implícitos escritos de distintas formas; se agrupan en una sola categoría explícita. |
| 3 | `NumberSubmitters` | Valor 0 | 682 | **Descartadas junto con #1.** Son las variantes *no classification for the single variant*: ningún remitente las clasificó en forma individual. |
| 4 | `LastEvaluated` | null | 30.580 | **"Sin fecha".** Son variantes clasificadas cuya fecha de evaluación no se informó (faltante real). La columna se mantiene como tipo fecha con NULL y "Sin fecha" se materializa como miembro especial de la dimensión fecha en el DW. |
| 5 | `RS# (dbSNP)` | Valor -1 | 1.681.084 (36,7 %) | **Columna descartada.** El -1 indica que la variante no está registrada en dbSNP (en CNV, inversiones y translocaciones ocurre en el 100 %). Es un identificador externo, no se puede imputar y no se usa en el análisis. |
| 6 | `GeneID`, `GeneSymbol` | `GeneID = -1` | 28.151 | **Filas descartadas.** Son variantes sin un gen único asignado: genes superpuestos (`KCNQ1;KCNQ1OT1`, 17.370), CNV grandes que abarcan muchos genes (*covers N genes…*, 4.551), símbolos sin ID (4.577) e intergénicas sin símbolo (1.653). `GeneID` es un identificador entero de NCBI y no admite etiquetas, y la tabla de genes necesita un ID válido como clave. Se pierde el 0,65 % de las filas. |
| 7 | `HGNC_ID` | null | 354 | **"Sin Datos".** Son genes `LOC…` sin nombre oficial en HGNC: el ID no existe, no es un dato que falte cargar. |
| 8 | `Cytogenetic` | null en el cromosoma MT | 3.058 | **"MT".** El ADN mitocondrial no tiene bandas citogenéticas; el faltante es estructural. |
| 9 | `Cytogenetic` | null fuera de MT | 3 | **Filas descartadas.** Faltantes reales sin forma confiable de imputarlos; su peso es despreciable. |
| 10 | `Chromosome` | "Un" (*unplaced*) | 2 | **Se conserva en esta etapa.** Es una categoría válida del genoma de referencia, no un faltante. Se descarta en la 1NF (ver Transformación a 1NF). |
| 11 | `PhenotypeList` | null, *not provided*, *not specified*, *See cases* | 2.916.570 condiciones | **"not provided" con un único ID MedGen (C3661900).** ClinVar usa tres IDs distintos para "no informado" (C3661900, CN517202, CN169374); se unifican en uno. *See cases* indica que el fenotipo se cargó por paciente y no a nivel de la variante, así que a este nivel equivale a "no informado".|
| 12 | `PhenotypeList` | "N conditions" (ej. *6 conditions*) | 7.269 condiciones | **"multiple conditions" (clave `MULTIPLE`).** Es una etiqueta resumen sin ID ni detalle de cuáles son las condiciones. |
| 13 | `PhenotypeIDS` | Condición con nombre pero sin ID MedGen | 177.495 condiciones | **"Sin ID MedGen" (clave `NO_MEDGEN`).** La clave de fenotipos es el ID MedGen (corto, estable y con un único nombre por ID); los ~10.000 nombres sin ID se agrupan en este miembro especial. Así la variante conserva su vínculo con un fenotipo, aunque se pierde el nombre específico. Afecta a 174.570 variantes. |
| 14 | `PhenotypeList`, `PhenotypeIDS` | Columnas multivalor (`\|` separa registros, `;` condiciones, `,` IDs de distintas ontologías) | Todas | **Reemplazadas por `PhenotypeKeys`**, una lista sin repetidos de claves por variante (ID MedGen, `C3661900`, `MULTIPLE` o `NO_MEDGEN`), y por la tabla `fenotipos` (clave → nombre, 11.731 fenotipos). Los ~350 nombres que traen `;` adentro se resuelven con el nombre completo cuando el registro tiene un solo ID. |
| 15 | `OriginSimple` | *not applicable* | 56 | **Se conserva.** Ya es una categoría explícita. |
| 16 | `OriginSimple` | *unknown* | 48.963 | **Unificado en "not provided".** *unknown* y *not provided* expresan lo mismo: el origen no fue informado. |
| 17 | `Origin` | Multivalor (`;`) | — | **Se conserva.** Contiene los mismos faltantes que `OriginSimple`, combinados; se resuelve en la normalización. |


## Aplicación de las decisiones

In [21]:
TEXTOS_SIN_CLASIF = SIN_CLASIF + ["no classification provided"]

df_limpio = (df
    .filter(pl.col("ClinSigSimple") != -1,  # 1 y 3
            pl.col("GeneID") != -1)         # 6
    .with_columns(
        # 2
        *[pl.when(pl.col(c).cast(pl.String).is_in(TEXTOS_SIN_CLASIF)).then(pl.lit("Sin clasificación"))
            .otherwise(pl.col(c).cast(pl.String))
            .cast(pl.Categorical).alias(c)
          for c in ["ClinicalSignificance", "ReviewStatus"]],
        # 7
        pl.col("HGNC_ID").fill_null("Sin Datos"),
        # 8
        pl.when(pl.col("Chromosome") == "MT").then(pl.lit("MT"))
          .otherwise(pl.col("Cytogenetic").cast(pl.String)).cast(pl.Categorical).alias("Cytogenetic"),
        # 16
        pl.col("OriginSimple").cast(pl.String).replace("unknown", "not provided").cast(pl.Categorical),
    )
    .filter(pl.col("Cytogenetic").is_not_null()))  # 9

print(f"Filas: {df.height:,} -> {df_limpio.height:,} (descartadas: {df.height - df_limpio.height})")
df_limpio.null_count()

Filas: 4,578,190 -> 4,304,792 (descartadas: 273398)


#AlleleID,VariationID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Chromosome,Cytogenetic,Start,Stop,ReviewStatus,NumberSubmitters
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,30580,59905,85,0,0,0,0,0,0,0,0


In [22]:
# 11-14: cada condición (separadas por | y ;) se resuelve a una clave MedGen
NOT_PROVIDED = "C3661900"
NP_IDS = ["C3661900", "CN517202", "CN169374"]
NP_NOMBRES = ["not provided", "not specified", "See cases"]

df_limpio = df_limpio.with_row_index("fila")

condiciones = (df_limpio
    .select("fila",
            pl.col("PhenotypeList").fill_null("not provided").str.split("|").alias("nom"),
            pl.col("PhenotypeIDS").str.split("|").alias("ids"))
    # sin IDs: un string vacío por registro, para que quede alineado con los nombres
    .with_columns(pl.col("ids").fill_null(pl.col("nom").list.eval(pl.lit(""))))
    .explode("nom", "ids")                                   # nivel 1: registros (|)
    .with_columns(
        pl.col("nom").str.split(";").alias("noms"),
        pl.when(pl.col("ids") == "").then(pl.col("nom").str.split(";").list.eval(pl.lit("")))
          .otherwise(pl.col("ids").str.split(";")).alias("ids"))
    # ~350 nombres traen ';' adentro y no se alinean con sus IDs:
    # con un solo ID el nombre completo es el de esa condición; si no, se conservan los IDs sin nombre
    .with_columns(pl.when(pl.col("noms").list.len() == pl.col("ids").list.len()).then(pl.col("noms"))
                    .when(pl.col("ids").list.len() == 1).then(pl.concat_list("nom"))
                    .otherwise(pl.col("ids").list.eval(pl.lit(None, dtype=pl.String))).alias("nom"))
    .drop("noms")
    .explode("nom", "ids")                                   # nivel 2: condiciones (;)
    .with_columns(pl.col("ids").str.extract(r"MedGen:([A-Z0-9]+)").alias("medgen"))
    .with_columns(
        pl.when(pl.col("nom").is_in(NP_NOMBRES) | pl.col("medgen").is_in(NP_IDS)).then(pl.lit(NOT_PROVIDED))
          .when(pl.col("nom").str.contains(r"^\d+ conditions$")).then(pl.lit("MULTIPLE"))
          .when(pl.col("medgen").is_not_null()).then(pl.col("medgen"))
          .otherwise(pl.lit("NO_MEDGEN")).alias("clave")))

fenotipos = pl.concat([
    pl.DataFrame({"clave": [NOT_PROVIDED, "MULTIPLE", "NO_MEDGEN"],
                  "nombre": ["not provided", "multiple conditions", "Sin ID MedGen"]}),
    condiciones.filter(~pl.col("clave").is_in([NOT_PROVIDED, "MULTIPLE", "NO_MEDGEN"]))
               .select("clave", pl.col("nom").alias("nombre")).drop_nulls().unique(),
])

df_limpio = (df_limpio
    .join(condiciones.group_by("fila").agg(pl.col("clave").unique().sort().alias("PhenotypeKeys")), on="fila")
    .sort("fila")
    .drop("fila", "PhenotypeList", "PhenotypeIDS"))

print(f"Condiciones: {condiciones.height:,} | fenotipos distintos: {fenotipos.height:,}")
condiciones["clave"].replace_strict({NOT_PROVIDED: "not provided", "MULTIPLE": "multiple conditions",
                                     "NO_MEDGEN": "Sin ID MedGen"}, default="MedGen real").value_counts(sort=True)

Condiciones: 6,222,822 | fenotipos distintos: 11,731


clave,count
str,u32
"""MedGen real""",3121488
"""not provided""",2916570
"""Sin ID MedGen""",177495
"""multiple conditions""",7269


# Outliers

Las variables numéricas (largo de la variante, `NumberSubmitters`, `LastEvaluated`) tienen valores extremos, pero son biológicamente legítimos: CNV e inversiones de varios Mb, variantes fundadoras informadas por decenas de laboratorios (ej. GJB2 c.35del, 94 remitentes) y evaluaciones históricas de literatura. No se detectaron valores imposibles (Stop < Start, posiciones fuera del cromosoma, fechas futuras). Se decide conservarlos sin transformar.

# Normalización

## 1NF

### Requisitos de la 1NF

| # | Requisito | Qué significa |
|---|---|---|
| 1 | Valores atómicos | Cada celda guarda un único valor indivisible, no listas ni valores separados por un delimitador. |
| 2 | Un solo tipo por columna | Todos los valores de una columna pertenecen al mismo dominio y tipo de dato. |
| 3 | Sin grupos repetidos | No guardar varios valores del mismo atributo en una fila, ni en una celda (lista) ni en columnas repetidas (`Fenotipo1`, `Fenotipo2`, …). |
| 4 | Clave primaria | Cada fila se identifica de forma única por una clave. |
| 5 | Sin filas duplicadas | No puede haber dos filas idénticas. Se deriva de #4: si la PK no se repite, ninguna fila puede repetirse. |
| 6 | Nombres de columna únicos | Cada columna tiene un nombre distinto. |

La redundancia (por ejemplo, repetir el símbolo de un gen en cada variante de ese gen) no viola la 1NF: se resuelve en la 2NF y la 3NF.

**Clave Primaria Elegida: `VariationID`**

### Transformación a 1NF

- **`PhenotypeKeys`** (lista) se separa en la tabla `variante_fenotipo`, con una fila por par variante–fenotipo.
- **`Origin`** (valores separados por `;`) se elimina: `OriginSimple` es atómica y resume el mismo dato.
- **`ClinicalSignificance`** con varias clasificaciones separadas por `;` (ej. `Pathogenic; drug response`): filas descartadas.
- **`Cytogenetic`** con dos bandas separadas por `;` (ej. `Xp22.33;Yp11.2`): filas descartadas. Son las variantes que aparecen en dos cromosomas a la vez (regiones pseudoautosómicas X/Y, translocaciones y loci duplicados).
- **`Chromosome = "Un"`** (*unplaced*): filas descartadas. Son secuencias del ensamblado sin cromosoma asignado; sus coordenadas son relativas a ese fragmento y no se pueden comparar con las del resto.

In [ ]:
df_1nf = df_limpio.filter(
    ~pl.col("ClinicalSignificance").cast(pl.String).str.contains(";"),
    ~pl.col("Cytogenetic").cast(pl.String).str.contains(";"),
    pl.col("Chromosome") != "Un")
print(f"Filas descartadas: {df_limpio.height - df_1nf.height:,}")

variante_fenotipo = (df_1nf
    .select("VariationID", pl.col("PhenotypeKeys").alias("PhenotypeKey"))
    .explode("PhenotypeKey"))

df_1nf = df_1nf.drop("PhenotypeKeys", "Origin")

print(f"df_1nf: {df_1nf.height:,} filas x {df_1nf.width} columnas")
print(f"variante_fenotipo: {variante_fenotipo.height:,} filas")

### Superclaves

Una **superclave** es una combinación de columnas que no se repite en ninguna fila. Se evalúa sobre `df_1nf`.

| Combinación | Valores distintos | Filas duplicadas | ¿Superclave? |
|---|---|---|---|
| `#AlleleID` | 4.302.694 | 0 | ✅ |
| `VariationID` | 4.302.694 | 0 | ✅ |
| `Name` (HGVS completo) | 4.302.407 | 303 | ❌ |
| Transcrito NM (extraído de `Name`) | 22.114 | 4.300.958 | ❌ |
| `Chromosome + Start + Stop + Type` | 3.914.685 | 736.221 | ❌ |
| Transcrito NM `+ Chromosome + Start + Stop + Type` | 3.915.460 | 734.914 | ❌ |
| `Name + Chromosome + Start` | 4.302.674 | 38 | ❌ |
| `#AlleleID + Chromosome` | 4.302.694 | 0 | ✅ |
| `VariationID + Chromosome` | 4.302.694 | 0 | ✅ |

Antes de la transformación a 1NF, `#AlleleID` y `VariationID` se repetían en 1.700 filas: la misma variante aparecía en dos cromosomas (regiones pseudoautosómicas X/Y, translocaciones, un locus en "Un"). Esas filas eran las que tenían dos bandas en `Cytogenetic` o cromosoma "Un", y se descartaron por no ser atómicas.

`Name` no es única porque distintas CNV pueden compartir la misma descripción (ej. `GRCh38/hg38 16p11.2-11.1(...)x3`). El transcrito NM no sirve como base de una clave: lo comparten todas las variantes de un gen, para distinguirlas harían falta los alelos de referencia y alternativo (descartados en la carga) y 21.791 filas no tienen NM (nombres `NC_`, `NR_` o CNV).

In [ ]:
COMBOS = [["#AlleleID"], ["VariationID"], ["Name"],
          ["Chromosome", "Start", "Stop", "Type"],
          ["#AlleleID", "Chromosome"], ["VariationID", "Chromosome"]]

# Se evalúa una combinación por vez para no cargar la memoria
pl.DataFrame([{"combinacion": " + ".join(cs),
               "filas_duplicadas": df_1nf.select(cs).is_duplicated().sum()}
              for cs in COMBOS]).with_columns((pl.col("filas_duplicadas") == 0).alias("superclave"))

### Claves candidatas

Una **clave candidata** es una superclave mínima: si se le quita cualquier columna, deja de ser única.

| Superclave | ¿Mínima? | ¿Candidata? |
|---|---|---|
| `#AlleleID` | ✅ Una sola columna | ✅ |
| `VariationID` | ✅ Una sola columna | ✅ |
| `#AlleleID + Chromosome` | ❌ `Chromosome` sobra: `#AlleleID` ya es única | ❌ |
| `VariationID + Chromosome` | ❌ `Chromosome` sobra: `VariationID` ya es única | ❌ |

### Clave primaria: `VariationID`

- `VariationID` es el identificador público de ClinVar (registro VCV, URL `clinvar/variation/{VariationID}`), y la clasificación clínica que se analiza corresponde a ese registro.
- `#AlleleID` queda como clave alternativa (`UNIQUE`).
- Una PK de una sola columna es más simple de referenciar desde otras tablas (`variante_fenotipo`) y, al no ser compuesta, no admite dependencias parciales.

### Filas duplicadas

Si la clave primaria no se repite, tampoco puede haber dos filas idénticas en todas sus columnas. Verificarlo sobre la PK evita comparar las 17 columnas, que con textos largos agota la memoria.

In [ ]:
dups = df_1nf["VariationID"].is_duplicated().sum()
print(f"Filas duplicadas: {dups}")

## 2NF

Una tabla está en 2NF si está en 1NF y ninguna columna que no es clave depende de solo una parte de la clave primaria (dependencia parcial). Eso solo puede pasar cuando la PK es compuesta.

La PK de `df_1nf` es una sola columna (`VariationID`), así que no hay dependencias parciales posibles: la tabla ya cumple 2NF.